# Module 03 — Autograd and Computational Graphs

**Prerequisites:** Module 02 (Tensor Operations & Broadcasting)
**Time:** ~100 minutes

## Learning Objectives

- Explain what a gradient is and why training a neural network needs one.
- Explain what a computational graph is and how PyTorch builds one automatically.
- Use `requires_grad`, `.backward()`, and `.grad` to compute gradients.
- Explain why gradients accumulate by default, and why `zero_grad()` exists.
- Use `torch.no_grad()` correctly, and explain what it turns off.
- Use `detach()` to break tensors out of the computation graph.
- Walk through the chain rule step by step and verify it with autograd.
- Understand gradient clipping at a conceptual level.


In [ ]:
import torch


## Why Do We Need Gradients?

Training a neural network means: gradually adjusting its parameters (weights) so its predictions get better. "Better" is measured by a **loss function** — a single number that's high when predictions are bad and low when they're good.

The question training needs answered, over and over, is: *if I nudge this particular weight up slightly, does the loss go up or down, and by how much?* That sensitivity — "how much does the loss change per tiny change in this weight" — is exactly what a **derivative** (a **gradient**, once you have many weights) tells you.

Once you know the gradient of the loss with respect to every weight, you can nudge every weight in the direction that *decreases* the loss. Repeat thousands of times, and the model gets better. This process is called **gradient descent**, and it's the algorithm underneath essentially every neural network ever trained.

The problem: a real model might have millions of weights, connected through dozens of layers of arithmetic. Computing every one of those derivatives by hand (as you'd do with pen-and-paper calculus, using the chain rule repeatedly) is completely impractical. **Autograd is PyTorch's system for computing all of these derivatives automatically**, and it's arguably the single most important feature PyTorch provides.


## A Tiny Example, By Hand First

Let's use the smallest possible example so the calculus is trivial, and compare "doing it by hand" to "letting autograd do it."

Suppose $z = x^2$. From calculus, $\frac{dz}{dx} = 2x$. If $x = 3$, then $\frac{dz}{dx} = 6$.


In [ ]:
x = torch.tensor(3.0, requires_grad=True)   # requires_grad=True: "track operations on this tensor"
z = x ** 2

z.backward()   # compute d(z)/d(x) for every tensor that requires_grad, working backward from z

print("x.grad:", x.grad)   # should be 2 * 3 = 6, matching the hand calculation


That matched our hand calculation: `2 * 3 = 6`. Now let's see what's actually happening.

**`requires_grad=True`** tells PyTorch: "remember every operation applied to this tensor, because I'll want gradients with respect to it later." Tensors created this way (or derived from such tensors) are part of a **computational graph** — a record of every operation, connecting inputs to outputs.

**`z.backward()`** walks that graph *backward*, starting from `z`, applying the chain rule at every step, until it reaches every `requires_grad=True` tensor that contributed to `z`. At each one, it stores the computed derivative in that tensor's `.grad` attribute.

This is called **tape-based** (or "reverse-mode") automatic differentiation: PyTorch records a "tape" of operations during the forward pass, then replays it backward to compute gradients.


## Visualizing the Computational Graph

When you write `z = x ** 2`, PyTorch internally builds a graph:

```
x (leaf, requires_grad=True)
│
▼
[PowBackward0]  (operation: x ** 2)
│
▼
z (result, grad_fn=PowBackward0)
```

When you call `z.backward()`, PyTorch walks this graph in reverse:
1. Starts at `z`
2. Asks: "What operation produced `z`?" → `PowBackward0` (squaring)
3. Applies the derivative of squaring: $\frac{d}{dx}(x^2) = 2x$
4. Stores `2 * 3 = 6` in `x.grad`

Let's verify this by looking at `z.grad_fn`:


In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = x * 2
z = y ** 2 + y

# Each intermediate tensor remembers what operation created it
print("x.grad_fn:", x.grad_fn)    # None — x is a leaf (created directly, not computed)
print("y.grad_fn:", y.grad_fn)    # MulBackward0
print("z.grad_fn:", z.grad_fn)    # AddBackward0
print("x.is_leaf:", x.is_leaf)    # True
print("z.is_leaf:", z.is_leaf)    # False


The computational graph for `z = (x*2)^2 + (x*2)` looks like:

```
x (leaf)
│
▼
[MulBackward0]  x * 2 = y
│          │
▼          ▼
[PowBackward0]  y^2     [identity]  y
│                       │
└───────────┬───────────┘
            ▼
    [AddBackward0]  y^2 + y = z
```

`.backward()` walks this whole graph in reverse, applying the chain rule at each node.


## The Chain Rule, Step by Step

Let's verify the chain rule works for a compound expression: $z = (2x)^2 + 2x$.

**By hand:**
- Let $y = 2x$, so $z = y^2 + y$
- $\frac{dz}{dy} = 2y + 1$
- $\frac{dy}{dx} = 2$
- By the chain rule: $\frac{dz}{dx} = \frac{dz}{dy} \cdot \frac{dy}{dx} = (2y + 1) \cdot 2 = (2 \cdot 2x + 1) \cdot 2$
- At $x = 3$: $(2 \cdot 6 + 1) \cdot 2 = 13 \cdot 2 = 26$


In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = 2 * x
z = y ** 2 + y

z.backward()
print("x.grad:", x.grad)   # should be 26

# Manual verification
manual = (2 * (2 * 3) + 1) * 2
print("manual:", manual)


### 🔮 Predict before you run

For a *vector* input `x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)` and `z = (x ** 2).sum()`, what do you expect `x.grad` to be? (Hint: the derivative of $\sum x_i^2$ with respect to each $x_i$ is $2x_i$ — apply this elementwise.)


In [ ]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
z = (x ** 2).sum()
z.backward()
print("x.grad:", x.grad)   # expect [2, 4, 6]


`.backward()` requires the tensor you call it on (`z` here) to be a **scalar** (a single number) — that's why we called `.sum()` first. This makes sense conceptually: "the gradient of a vector with respect to another vector" is more complex than a single number's worth of information, and in the overwhelming majority of deep learning, the thing you call `.backward()` on is the loss — always a single scalar number.


## Three Rules of Autograd

1. **Only leaf tensors with `requires_grad=True` accumulate gradients into `.grad`.** A "leaf" tensor is one you created directly (not the result of an operation on other tracked tensors). Model parameters (`nn.Linear`'s weights, for example) are leaf tensors with `requires_grad=True` set automatically — you'll see this in the next module.
2. **Gradients accumulate (add up) by default across multiple `.backward()` calls**, rather than being overwritten. This is intentional — it supports advanced use cases — but it means you must explicitly reset gradients to zero before each new backward pass in a training loop, or old gradients will corrupt the new computation.
3. **`torch.no_grad()` disables gradient tracking** for a block of code — used whenever you don't need gradients (evaluation, inference), because tracking has both a memory and compute cost.

Let's see rule 2 in action, since it's the one that causes real bugs.


In [ ]:
x = torch.tensor(2.0, requires_grad=True)

z1 = x ** 2
z1.backward()
print("after first backward:", x.grad)   # 2*2 = 4

z2 = x ** 2
z2.backward()
print("after second backward (no zeroing!):", x.grad)   # NOT 4 again -- it's 4+4=8!


This is exactly why every training loop you'll write contains `optimizer.zero_grad()` at the start of each iteration — without it, gradients from *every previous batch* silently pile up on top of each other, and the model updates itself using wrong, inflated gradients. Let's fix the example above:


In [ ]:
x = torch.tensor(2.0, requires_grad=True)

z1 = x ** 2
z1.backward()
print("after first backward:", x.grad)

x.grad.zero_()   # reset gradient to 0 before the next backward pass
z2 = x ** 2
z2.backward()
print("after zeroing + second backward:", x.grad)   # correctly 4 again


## `torch.no_grad()`: Turning Tracking Off

Every operation on a `requires_grad=True` tensor costs extra memory (to store the computational graph) and extra compute (to enable `.backward()` later). During evaluation or inference — when you'll never call `.backward()` — this tracking is pure waste.


In [ ]:
x = torch.tensor(3.0, requires_grad=True)

with torch.no_grad():
    y = x ** 2   # this operation is NOT tracked

print("y.requires_grad:", y.requires_grad)   # False, even though x requires grad!

# Trying to call .backward() on y would raise an error, since no graph was recorded:
try:
    y.backward()
except RuntimeError as e:
    print("Error:", e)


In [ ]:
# The decorator form — equivalent to wrapping the entire function body in with torch.no_grad():
@torch.no_grad()
def predict(x):
    return x ** 2 + 3 * x + 1

x = torch.tensor(5.0, requires_grad=True)
result = predict(x)
print("result:", result)
print("requires_grad:", result.requires_grad)   # False


You'll see `with torch.no_grad():` wrapped around every evaluation loop and every inference call in this course, for exactly this reason.


## `detach()`: Breaking the Graph

Sometimes you want to use a tensor's *value* without its gradient history. `detach()` returns a new tensor that shares the same data but is disconnected from the computational graph.


In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2

# y is connected to the graph — it remembers it came from x**2
print("y.grad_fn:", y.grad_fn)

# detach() breaks the connection
y_detached = y.detach()
print("y_detached.grad_fn:", y_detached.grad_fn)   # None
print("y_detached.requires_grad:", y_detached.requires_grad)   # False
print("y_detached value:", y_detached)   # same value, just not tracked


**When you'll use `detach()`:**

1. **Stopping gradient flow** — e.g., in a GAN, you might detach the generator's output before passing it to the discriminator, so gradients don't flow back to the generator during the discriminator's training step.
2. **Converting to NumPy** — `.numpy()` requires a tensor without gradient tracking: `tensor.detach().cpu().numpy()`.
3. **Using a value as a fixed target** — e.g., in self-supervised learning or target networks.


In [ ]:
# Common pattern: converting model output to NumPy for plotting/logging
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x ** 2

# This would fail:
try:
    y.numpy()
except RuntimeError as e:
    print("Error:", e)

# This works:
y_np = y.detach().numpy()
print("as numpy:", y_np)


## `retain_graph=True` (Advanced)

By default, calling `.backward()` **destroys** the computational graph after using it — this saves memory. If you need to call `.backward()` multiple times on the same graph (rare, but happens in some research settings), pass `retain_graph=True`.


In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 3

# First backward: works fine
y.backward(retain_graph=True)   # keep the graph alive
print("first backward:", x.grad)   # 3 * 2^2 = 12

# Second backward on the same graph: only works because we retained it
x.grad.zero_()
y.backward()   # graph is destroyed after this call (default behavior)
print("second backward:", x.grad)   # 12 again

# Third backward would fail:
try:
    x.grad.zero_()
    y.backward()
except RuntimeError as e:
    print("Error (graph already freed):", str(e)[:80])


## Higher-Order Gradients (Concept)

Sometimes you need the gradient *of a gradient* — a second derivative. This is used in:
- **Meta-learning** (MAML) — gradient of the loss of the inner-loop gradient update
- **Physics-informed neural networks** — enforcing PDE constraints on model outputs

PyTorch supports this via `create_graph=True` in `.backward()` or `torch.autograd.grad()`.


In [ ]:
# Second derivative: d²(x³)/dx² = 6x, at x=2: 12
x = torch.tensor(2.0, requires_grad=True)
y = x ** 3

# First derivative using torch.autograd.grad (function-style API)
dy_dx = torch.autograd.grad(y, x, create_graph=True)[0]
print("dy/dx:", dy_dx)   # 3 * 2^2 = 12

# Second derivative: differentiate the first derivative
d2y_dx2 = torch.autograd.grad(dy_dx, x)[0]
print("d²y/dx²:", d2y_dx2)   # 6 * 2 = 12


## Gradient Clipping (Preview)

In very deep networks or recurrent networks, gradients can become extremely large ("exploding gradients"), causing the model to diverge. **Gradient clipping** caps the gradient norm to a maximum value before the optimizer applies the update.

```python
# In a training loop, after loss.backward() and before optimizer.step():
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
```

This rescales all gradients so their combined norm doesn't exceed `max_norm`. We'll use this in later notebooks when training deeper models.


In [ ]:
# Demonstrating the concept: manually clipping a gradient
x = torch.tensor(100.0, requires_grad=True)
y = x ** 2
y.backward()

print("original gradient:", x.grad)   # 200 — very large!

# Manual clipping (in practice, use torch.nn.utils.clip_grad_norm_)
max_norm = 10.0
grad_norm = x.grad.norm()
if grad_norm > max_norm:
    x.grad.data.mul_(max_norm / grad_norm)
print("clipped gradient:", x.grad)     # scaled down to norm=10


## 🐛 Debugging Challenge

The code below is supposed to compute the gradient of `loss` with respect to `w`, but it fails. Find the bug before reading the explanation.


In [ ]:
w = torch.tensor(5.0)   # note: no requires_grad!
target = torch.tensor(10.0)

prediction = w * 2
loss = (prediction - target) ** 2

try:
    loss.backward()
    print(w.grad)
except RuntimeError as e:
    print("Error:", e)


**Diagnosis:** `w` was created without `requires_grad=True`, so PyTorch never tracked any operations involving it — there's no graph connecting `loss` back to `w`, so there's nothing to walk backward through.

**Fix:** set `requires_grad=True` when creating `w`.


In [ ]:
w = torch.tensor(5.0, requires_grad=True)
target = torch.tensor(10.0)

prediction = w * 2
loss = (prediction - target) ** 2
loss.backward()
print(w.grad)   # d(loss)/dw = 2*(2w - 10)*2 = 4*(2*5-10) = 0 here, since prediction already equals target


### 🐛 Debugging Challenge 2: Mysterious NaN Gradients

Sometimes gradients become `NaN` (Not a Number). This usually means an operation produced an undefined result somewhere in the graph. Can you spot the issue?


In [ ]:
x = torch.tensor(0.0, requires_grad=True)
y = torch.sqrt(x)   # sqrt(0) = 0, but d/dx sqrt(x) = 1/(2*sqrt(x)) which is 1/0 at x=0!

y.backward()
print("gradient at x=0:", x.grad)   # inf or nan — the derivative is undefined at x=0

# Fix: add a small epsilon to avoid division by zero
x2 = torch.tensor(0.0, requires_grad=True)
y2 = torch.sqrt(x2 + 1e-8)  # epsilon prevents the singularity
y2.backward()
print("gradient with epsilon:", x2.grad)   # large but finite


## Implementing Manual Gradient Descent

Let's put it all together: implement one complete gradient descent optimization on $f(x) = (x-5)^2$, which has its minimum at $x = 5$.


In [ ]:
x = torch.tensor(0.0, requires_grad=True)
learning_rate = 0.1

print(f"{'Step':>4} | {'x':>8} | {'f(x)':>8} | {'grad':>8}")
print("-" * 40)

for step in range(20):
    # Forward: compute the function
    loss = (x - 5) ** 2
    
    # Backward: compute the gradient
    loss.backward()
    
    if step % 4 == 0:
        print(f"{step:4d} | {x.item():8.4f} | {loss.item():8.4f} | {x.grad.item():8.4f}")
    
    # Update: nudge x in the direction that decreases loss
    with torch.no_grad():
        x -= learning_rate * x.grad
    
    # Zero the gradient for the next iteration
    x.grad.zero_()

print(f"\nFinal x = {x.item():.4f} (target was 5.0)")


## Exercises

🟢 **Beginner 1:** Create `x = torch.tensor(4.0, requires_grad=True)`. Compute `y = 3 * x + 1`, call `.backward()`, and check that `x.grad` equals 3 (the coefficient — since $y = 3x + 1$ has constant slope 3).

🟢 **Beginner 2:** Create a tensor with `requires_grad=True`, compute some operation on it, then check `result.grad_fn` and `result.is_leaf`. Explain what you see.

🟡 **Intermediate 1:** Create `x = torch.tensor(2.0, requires_grad=True)`. In a loop that runs 3 times: compute `y = x ** 2`, call `.backward()`, and print `x.grad` *without* zeroing it between iterations. Confirm the values are `4, 8, 12` (accumulating), then repeat the loop but zero the gradient each iteration and confirm you instead get `4, 4, 4`.

🟡 **Intermediate 2:** Create a vector `x = torch.tensor([1.0, 2.0, 3.0, 4.0], requires_grad=True)`. Compute `z = (x ** 3).sum()` and find the gradient. Verify by hand: $\frac{d}{dx_i}(x_i^3) = 3x_i^2$.

🔴 **Challenge 1:** Implement gradient descent on the Rosenbrock function $f(x, y) = (1-x)^2 + 100(y-x^2)^2$ starting from `(x, y) = (-1.0, 1.0)` with `lr=0.001` for 5000 steps. The minimum is at `(1, 1)`. Print the final `(x, y)` values.

🔴 **Challenge 2:** Use `torch.autograd.grad()` to compute both the first and second derivatives of $f(x) = x^4$ at $x = 3$. Verify: $f'(x) = 4x^3 = 108$, $f''(x) = 12x^2 = 108$.


In [ ]:
# Space for your exercise solutions



## Common Mistakes

- **Forgetting `requires_grad=True`** on a tensor you intend to compute gradients for, then being confused when `.grad` is `None`.
- **Forgetting to zero gradients** between training steps, causing gradients to silently accumulate across batches.
- **Calling `.backward()` on a non-scalar tensor** without reducing it to a single number first (usually via `.sum()` or `.mean()`).
- **Forgetting `torch.no_grad()`** during evaluation/inference, wasting memory and compute.
- **Trying to call `.numpy()` on a tensor with gradients** — use `.detach().numpy()` instead.
- **NaN gradients** from operations with undefined derivatives (e.g., `sqrt(0)`, `log(0)`) — add epsilon.

## Mental Model

Think of every `requires_grad=True` tensor as carrying an invisible "how did I get here?" tag. As you compute `y = x**2`, then `z = y + 3`, PyTorch is silently building a chain: `x → y → z`. Calling `z.backward()` walks that chain backward, applying the chain rule at every link, and drops the final answer into `x.grad`. `torch.no_grad()` is a way of saying "don't bother tagging anything in here — I won't need the history." `detach()` cuts one specific tensor out of the chain while keeping everything else connected.

## Key Takeaways

- A gradient tells you how much a small change in one number affects another — training uses this to know which direction to adjust weights.
- `requires_grad=True` + `.backward()` is PyTorch's automatic differentiation system, replacing hand-derived calculus.
- PyTorch builds a computational graph during the forward pass and walks it backward during `.backward()`.
- Gradients accumulate by default — always zero them before the next backward pass in a training loop.
- `torch.no_grad()` disables tracking for evaluation/inference, saving memory and compute.
- `detach()` disconnects a tensor from the graph — essential for NumPy conversion and stopping gradient flow.
- Gradient clipping prevents exploding gradients in deep networks.

## What's Next

**Module 04 — Building Neural Networks: From Scratch to `nn.Module`** uses everything from this notebook to build, train, and then refactor a tiny model — first with raw tensors and autograd, then using PyTorch's `nn.Linear` and `nn.Module` abstractions.

## Checklist

- [ ] I can explain in my own words why training needs gradients
- [ ] I can compute a gradient with `requires_grad=True` and `.backward()` and verify it against hand calculus
- [ ] I can draw a simple computational graph and explain how `.backward()` traverses it
- [ ] I understand why gradients accumulate and why `zero_grad()`/`.grad.zero_()` is necessary
- [ ] I know when and why to use `torch.no_grad()`
- [ ] I know what `detach()` does and when to use it
- [ ] I can walk through the chain rule for a multi-step computation and verify with autograd
